swin transformer


In [ ]:
# ======================================================
# Swin Transformer - Fine-Tune (CLASSIFIER ONLY)
# Save: .pkl (Streamlit Ready)
# ======================================================

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from timm import create_model
import joblib
from tqdm import tqdm
import pickle

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", device)

BASE_DIR = r"D:\alzheimer detection.v1i.folder\processed_dataset"
SAVE_DIR = r"D:\alzheimer detection.v1i.folder\Dashboard\src\Transformer\model"
os.makedirs(SAVE_DIR, exist_ok=True)
MODEL_PATH = os.path.join(SAVE_DIR, "swin_finetune_best.pkl")
METRICS_PATH = os.path.join(SAVE_DIR, "swin_finetune_metrics.pkl")

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 4
EPOCHS = 20
PATIENCE = 3
LR = 1e-4

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5],[0.5, 0.5, 0.5])
])

train_ds = datasets.ImageFolder(os.path.join(BASE_DIR, "train"), transform=transform)
val_ds   = datasets.ImageFolder(os.path.join(BASE_DIR, "test"), transform=transform)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
class_names = train_ds.classes
print("Classes:", class_names)

# ---------------- MODEL ----------------
swin_model = create_model("swin_tiny_patch4_window7_224", pretrained=True)
swin_model.head = nn.Linear(swin_model.head.in_features, NUM_CLASSES)
swin_model = swin_model.to(device)

# Freeze backbone, train head only
for name, param in swin_model.named_parameters():
    if "head" not in name:
        param.requires_grad = False
    else:
        param.requires_grad = True

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, swin_model.parameters()), lr=LR)

# ---------------- TRAINING ----------------
best_val_acc = 0
patience_counter = 0
train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(EPOCHS):
    swin_model.train()
    running_loss, correct, total = 0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        features = swin_model.forward_features(images)
        outputs = swin_model.head(features)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = outputs.max(1)
        total += labels.size(0)
        correct += preds.eq(labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total

    # Validation
    swin_model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            features = swin_model.forward_features(images)
            outputs = swin_model.head(features)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, preds = outputs.max(1)
            val_total += labels.size(0)
            val_correct += preds.eq(labels).sum().item()

    val_loss /= len(val_loader)
    val_acc = val_correct / val_total

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"[Epoch {epoch+1}] Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        artifact = {
            "model_state_dict": swin_model.state_dict(),
            "architecture": "Swin Tiny (Fine-Tune Head)",
            "num_classes": NUM_CLASSES,
            "class_names": class_names,
            "img_size": IMG_SIZE,
            "normalization": {"mean":[0.5]*3, "std":[0.5]*3},
            "best_val_acc": best_val_acc,
            "epoch": epoch+1
        }
        joblib.dump(artifact, MODEL_PATH)
        print(f">>> Best model saved: {MODEL_PATH}")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

# Save metrics
metrics = {"train_losses": train_losses, "val_losses": val_losses, "train_accs": train_accs, "val_accs": val_accs}
with open(METRICS_PATH, "wb") as f:
    pickle.dump(metrics, f)

print("\n[SUCCESS] Swin Transformer Fine-Tune model ready for Streamlit")
print("Best Val Acc:", best_val_acc)
print("Model saved at:", MODEL_PATH)


DEVICE: cuda
Classes: ['Mild Impairment', 'Moderate Impairment', 'No Impairment', 'Very Mild Impairment']


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

Epoch 1/20:   0%|          | 0/184 [00:17<?, ?it/s]


RuntimeError: only batches of spatial targets supported (3D tensors) but got targets of size: : [32]

EfficientFormer

In [ ]:
# ======================================================
# EfficientFormer - Fine-Tune (CLASSIFIER ONLY)
# Save: .pkl (Streamlit Ready)
# ======================================================

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from timm import create_model
import joblib
from tqdm import tqdm
import pickle

# ======================================================
# DEVICE
# ======================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", device)

# ======================================================
# PATH CONFIG
# ======================================================
BASE_DIR = r"D:\alzheimer detection.v1i.folder\processed_dataset"
SAVE_DIR = r"D:\alzheimer detection.v1i.folder\Dashboard\src\Transformer\model"
os.makedirs(SAVE_DIR, exist_ok=True)
MODEL_PATH = os.path.join(SAVE_DIR, "efficientformer_finetune_best.pkl")
METRICS_PATH = os.path.join(SAVE_DIR, "efficientformer_finetune_metrics.pkl")

# ======================================================
# CONFIG
# ======================================================
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 4
EPOCHS = 15
LR = 1e-4

# ======================================================
# DATA TRANSFORMS
# ======================================================
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

train_ds = datasets.ImageFolder(os.path.join(BASE_DIR, "train"), transform=transform)
val_ds   = datasets.ImageFolder(os.path.join(BASE_DIR, "test"), transform=transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

class_names = train_ds.classes
print("Classes:", class_names)

# ======================================================
# MODEL: EfficientFormer (Fine-Tune Head)
# ======================================================
efficient_model = create_model(
    "efficientformer_l3",
    pretrained=True,
    num_classes=NUM_CLASSES
).to(device)

# Freeze backbone, fine-tune classifier only
for name, param in efficient_model.named_parameters():
    if "head" not in name:
        param.requires_grad = False

# ======================================================
# LOSS & OPTIMIZER
# ======================================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, efficient_model.parameters()), lr=LR)
scaler = torch.cuda.amp.GradScaler()

# ======================================================
# TRAINING LOOP
# ======================================================
best_val_acc = 0.0
patience_counter = 0

train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(EPOCHS):
    # ---------------- TRAIN ----------------
    efficient_model.train()
    correct, total, running_loss = 0, 0, 0.0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            outputs = efficient_model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        _, preds = outputs.max(1)
        total += labels.size(0)
        correct += preds.eq(labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total

    # ---------------- VALIDATION ----------------
    efficient_model.eval()
    val_correct, val_total, val_loss = 0, 0, 0.0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            with torch.cuda.amp.autocast():
                outputs = efficient_model(images)
                loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, preds = outputs.max(1)
            val_total += labels.size(0)
            val_correct += preds.eq(labels).sum().item()

    val_loss /= len(val_loader)
    val_acc = val_correct / val_total

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"[Epoch {epoch+1}] Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")

    # ---------------- SAVE BEST (.pkl) ----------------
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0

        artifact = {
            "model_state_dict": efficient_model.state_dict(),
            "architecture": "EfficientFormer L3 (Fine-Tune Head)",
            "num_classes": NUM_CLASSES,
            "class_names": class_names,
            "img_size": IMG_SIZE,
            "normalization": {
                "mean": [0.5, 0.5, 0.5],
                "std": [0.5, 0.5, 0.5]
            },
            "best_val_acc": best_val_acc,
            "epoch": epoch + 1
        }

        joblib.dump(artifact, MODEL_PATH)
        print(f">>> Best model saved: {MODEL_PATH}")

    else:
        patience_counter += 1
        if patience_counter >= 3:
            print(">>> Early stopping triggered!")
            break

# Save metrics
metrics = {
    "train_losses": train_losses,
    "val_losses": val_losses,
    "train_accs": train_accs,
    "val_accs": val_accs
}
with open(METRICS_PATH, "wb") as f:
    pickle.dump(metrics, f)
print(f"Metrics saved to {METRICS_PATH}")

print("\n[SUCCESS] EfficientFormer Fine-Tune model ready for Streamlit")
print("Best Val Acc:", best_val_acc)
print("Model saved at:", MODEL_PATH)


DEVICE: cuda
Classes: ['Mild Impairment', 'Moderate Impairment', 'No Impairment', 'Very Mild Impairment']


C:\Users\acer\AppData\Local\Temp\ipykernel_45568\1786956323.py:78: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Epoch 1/15:   0%|          | 0/184 [00:00<?, ?it/s]C:\Users\acer\AppData\Local\Temp\ipykernel_45568\1786956323.py:98: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/15: 100%|██████████| 184/184 [01:12<00:00,  2.55it/s]
C:\Users\acer\AppData\Local\Temp\ipykernel_45568\1786956323.py:121: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


[Epoch 1] Train Acc: 0.5866, Val Acc: 0.6592
>>> Best model saved: D:\alzheimer detection.v1i.folder\Dashboard\src\Transformer\model\efficientformer_finetune_best.pkl


Epoch 2/15: 100%|██████████| 184/184 [01:15<00:00,  2.44it/s]


[Epoch 2] Train Acc: 0.6646, Val Acc: 0.6760
>>> Best model saved: D:\alzheimer detection.v1i.folder\Dashboard\src\Transformer\model\efficientformer_finetune_best.pkl


Epoch 3/15: 100%|██████████| 184/184 [01:16<00:00,  2.39it/s]


[Epoch 3] Train Acc: 0.6776, Val Acc: 0.6801
>>> Best model saved: D:\alzheimer detection.v1i.folder\Dashboard\src\Transformer\model\efficientformer_finetune_best.pkl


Epoch 4/15:   0%|          | 0/184 [00:00<?, ?it/s]

ViT B/16

In [ ]:
# ======================================================
# ViT B/16 Fine-Tuning (Streamlit-ready .pkl)
# ======================================================

import os
import pickle
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from torchvision import datasets, transforms
import timm
import joblib

# ======================================================
# DEVICE
# ======================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ======================================================
# PATH CONFIG
# ======================================================
BASE_DIR = r"D:\alzheimer detection.v1i.folder\processed_dataset"
TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR   = os.path.join(BASE_DIR, "val")

SAVE_DIR = r"D:\alzheimer detection.v1i.folder\Dashboard\src\Transformer\model"
os.makedirs(SAVE_DIR, exist_ok=True)
MODEL_PATH = os.path.join(SAVE_DIR, "vit_finetuned_best.pkl")

# ======================================================
# DATASET CONFIG
# ======================================================
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 4
NUM_WORKERS = 2
EPOCHS = 15
PATIENCE = 3
LR = 5e-5

# ======================================================
# TRANSFORM
# ======================================================
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

# ======================================================
# DATASET & DATALOADER
# ======================================================
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=transform)
val_dataset   = datasets.ImageFolder(VAL_DIR, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

class_names = train_dataset.classes
print("Classes:", class_names)

# ======================================================
# MODEL (PRETRAINED → FINETUNE)
# ======================================================
model = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

# ======================================================
# TRAINING LOOP
# ======================================================
best_val_acc = 0.0
trigger_times = 0

train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(EPOCHS):
    # ---------------- TRAIN ----------------
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = outputs.max(1)
        total += labels.size(0)
        correct += preds.eq(labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc  = correct / total

    # ---------------- VALIDATION ----------------
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, preds = outputs.max(1)
            val_total += labels.size(0)
            val_correct += preds.eq(labels).sum().item()

    val_loss /= len(val_loader)
    val_acc = val_correct / val_total

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"[Epoch {epoch+1}] Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    # ---------------- SAVE BEST (.pkl Streamlit-ready) ----------------
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        trigger_times = 0

        artifact = {
            "model_state_dict": model.state_dict(),
            "architecture": "ViT Base /16",
            "num_classes": NUM_CLASSES,
            "class_names": class_names,
            "img_size": IMG_SIZE,
            "normalization": {"mean": [0.5]*3, "std": [0.5]*3},
            "best_val_acc": best_val_acc,
            "epoch": epoch+1
        }

        joblib.dump(artifact, MODEL_PATH)
        print(f">>> Best model saved: {MODEL_PATH}")

    else:
        trigger_times += 1
        print(f"No improvement ({trigger_times}/{PATIENCE})")
        if trigger_times >= PATIENCE:
            print(">>> Early stopping triggered")
            break

print("\n[SUCCESS] ViT Base /16 fine-tuned model ready for Streamlit")
print(f"Best Val Acc: {best_val_acc}")
print(f"Model saved at: {MODEL_PATH}")
